In [12]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [13]:
import sys

# Google Colab 환경에서 실행 중인지 확인
if 'google.colab' in sys.modules:
    # debconf를 Noninteractive 모드로 설정
    !echo 'debconf debconf/frontend select Noninteractive' | \
    debconf-set-selections

    # fonts-nanum 패키지를 설치
    !sudo apt-get -qq -y install fonts-nanum

    # Matplotlib의 폰트 매니저 가져오기
    import matplotlib.font_manager as fm

    # 나눔 폰트의 시스템 경로 찾기
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])

    # 찾은 각 나눔 폰트를 Matplotlib 폰트 매니저에 추가
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

In [14]:
import torch
import umap
import hdbscan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import pairwise_distances, silhouette_score
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

from sklearn.manifold import TSNE
import umap

In [15]:
plt.rcParams['font.family'] = 'NanumGothic'
print(plt.rcParams['font.family'], plt.rcParams['font.size'])   # 폰트확인

In [16]:
# 상품명 40개 예시
product_names = [
    # 운전자보험
    "참좋은운전자보험1904", "프리미엄운전자종합보험", "더안심운전자플랜2101", "무배당안전운전자보험2001", "하이카운전자보험플러스",
    # 암보험 / 건강보험
    "행복한명품암보험", "아이러브건강보험1904", "무배당헬스케어암보험", "암치료집중보장플랜", "더든든건강플랜보험",
    # 종신보험
    "TOP클래스유니버설종신101종표", "스테이지6대건강종신보험", "평생든든보장종신보험", "라이프플랜종신보험무배당", "무배당통합사망보장보험",
    # 어린이/여성
    "튼튼아이보험1906", "프리미엄여성건강보험", "태아부터자녀까지자녀보험", "무배당맘편한어린이보험", "셀럽케어여성암보험",
    # 실손/상해
    "더안심실손의료비보험", "365OK실손보험", "상해보장특화플랜보험", "무배당입원특약형실손보험", "건강한생활실손보험",
    # 주택/화재
    "삼성아파트혼합형", "주택화재종합보험", "무배당내집안심화재보험", "부동산임대보장보험", "전세보증금안심보험",
    # 저축성/변액/혼합형
    "컨버전스보험1004", "행복저축플랜변액보험", "무배당연금저축보험2019", "투자플러스종합변액보험", "리치플랜복합형보험",
    # 기타
    "더든든한무배당교보통합이보험", "마이플랜하이브리드보험", "VIP헬스케어서비스특약형보험", "스마트라이프웰빙보험", "온가족행복통합보장보험"
]

# 모델 로딩 함수
def get_embedding(model_name, sentences, model_type="sentence-transformer"):
    if model_type == "sentence-transformer":
        model = SentenceTransformer(model_name)
        return model.encode(sentences, convert_to_numpy=True)

    elif model_type == "huggingface":
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Kanana 모델만 별도로 처리
        if "kanana" in model_name.lower():
            model = AutoModel.from_pretrained(model_name)
            model.eval()
            with torch.no_grad():
                inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
                pool_mask = inputs["attention_mask"]
                outputs = model(**inputs, pool_mask=pool_mask)

                # 일반적으로 CLS 토큰 기준 (Kanana 문서에 따라 조정 가능)
                embeddings = outputs.last_hidden_state[:, 0].cpu().numpy()
                return embeddings

        else:
            model = AutoModel.from_pretrained(model_name)
            model.eval()
            with torch.no_grad():
                inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
                outputs = model(**inputs)

                # 일반 HuggingFace 모델: CLS token 기준
                embeddings = outputs.last_hidden_state[:, 0].cpu().numpy()
                return embeddings

# 모델 리스트: (이름, HuggingFace 모델명, 로딩 방식)
model_list = [
    ("KURE-v1", "nlpai-lab/KURE-v1", "sentence-transformer"),
    ("KoSimCSE-roberta", "jhgan/ko-sroberta-multitask", "sentence-transformer"),
    ("KR-SBERT-V40K", "snunlp/KR-SBERT-V40K-klueNLI-augSTS", "sentence-transformer"),
    # ("Kanana", "kakaocorp/kanana-nano-2.1b-embedding", "huggingface"),
    ("E5", "intfloat/multilingual-e5-large", "sentence-transformer"),
    ("BGE-M3", "BAAI/bge-m3", "sentence-transformer")
]

# 저장용
model_scores = []

for model_name, model_id, load_type in model_list:
    print(f"Processing: {model_name}")
    embeddings = get_embedding(model_id, product_names, load_type)

    # 유사도 행렬
    cosine_sim = 1 - pairwise_distances(embeddings, metric='cosine')

    # HDBSCAN 클러스터링
    clusterer = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=2)
    labels = clusterer.fit_predict(embeddings)

    # 시각화
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(embeddings)

    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, palette="tab10", s=80)
    plt.title(f"UMAP Clustering - {model_name}")
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # Silhouette Score (군집 품질)
    sil_score = silhouette_score(embeddings, labels) if len(set(labels)) > 1 else -1
    model_scores.append((model_name, sil_score))

# 결과 정렬 출력
print("\n📊 Silhouette Score 비교 (높을수록 클러스터 품질 우수):")
for name, score in sorted(model_scores, key=lambda x: -x[1]):
    print(f"{name:<20} | Silhouette Score: {score:.4f}")


In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def evaluate_embedding_metrics(embeddings, threshold=0.85, topk=10):
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, 0)

    # MeanPairwiseSim
    mean_pairwise = sim_matrix[np.triu_indices_from(sim_matrix, k=1)].mean()

    # AvgSim@TopK
    avg_topk_sim = np.mean([np.sort(row)[-topk:].mean() for row in sim_matrix])

    # Coverage@threshold
    coverage = np.mean([np.sum(row >= threshold) / topk for row in sim_matrix])

    return {
        "MeanPairwiseSim": round(mean_pairwise, 4),
        f"AvgSim@Top{topk}": round(avg_topk_sim, 4),
        f"Coverage@{threshold}": round(coverage, 4)
    }

# 전체 모델 평가
results = []

for name, model_id, load_type in model_list:
    print(f"Evaluating: {name}")
    try:
        embeddings = get_embedding(model_id, product_names, load_type)
        metrics = evaluate_embedding_metrics(embeddings)
        results.append((name, metrics))
    except Exception as e:
        print(f"⚠️ {name} 실패: {e}")

# 결과 출력
print("\n📊 정량 비교 결과 (높을수록 좋음):\n")
print(f"{'Model':<25} | MeanPairSim | AvgSim@Top10 | Coverage@0.85")
print("-" * 70)
for name, metrics in results:
    print(f"{name:<25} | {metrics['MeanPairwiseSim']:<12} | {metrics['AvgSim@Top10']:<13} | {metrics['Coverage@0.85']:<14}")

In [19]:
# 예시: 앞서 평가한 모델 결과 리스트 (자동 생성된 결과 사용)
# → 아래 부분은 앞서 실행된 `results` 리스트를 그대로 사용
# results = [
#     ("KURE-v1", {'MeanPairwiseSim': 0.6721, 'AvgSim@Top10': 0.8427, 'Coverage@0.85': 0.7889}),
#     ("KoSimCSE-roberta", {'MeanPairwiseSim': 0.6482, 'AvgSim@Top10': 0.8191, 'Coverage@0.85': 0.7556}),
#     ...
# ]

# DataFrame 변환
records = []
for model, metrics in results:
    for metric_name, value in metrics.items():
        records.append({
            "Model": model,
            "Metric": metric_name,
            "Value": value
        })

df = pd.DataFrame(records)

# 그래프 스타일 설정
# sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))

# 시각화
sns.barplot(data=df, x="Model", y="Value", hue="Metric")
plt.title("Quantitative comparison of embedding models with product name", fontsize=14)
plt.xticks(rotation=30)
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [20]:
# 시각화 함수
def visualize_embedding_2D(embeddings, labels, method="umap", model_name=""):
    if method == "umap":
        reducer = umap.UMAP(n_components=2, random_state=42)
        reduced = reducer.fit_transform(embeddings)
    elif method == "tsne":
        reducer = TSNE(n_components=2, perplexity=5, random_state=42)
        reduced = reducer.fit_transform(embeddings)
    else:
        raise ValueError("Method must be 'umap' or 'tsne'.")

    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], s=100)
    for i, label in enumerate(labels):
        plt.text(reduced[i, 0] + 0.01, reduced[i, 1], label, fontsize=9)
    plt.title(f"{model_name} - {method.upper()} Embedding")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# 전체 모델에 대해 임베딩 및 시각화 수행
for model_display_name, model_id, load_type in model_list:
    print(f"\n🔍 Processing model: {model_display_name}")
    # embeddings = get_embedding(model_id, product_names, load_type)

    # t-SNE 시각화
    visualize_embedding_2D(embeddings, product_names, method="tsne", model_name=model_display_name)

In [21]:
# 전체 모델에 대해 임베딩 및 시각화 수행
for model_display_name, model_id, load_type in model_list:
    print(f"\n🔍 Processing model: {model_display_name}")
    # embeddings = get_embedding(model_id, product_names, load_type)

    # UMAP 시각화
    visualize_embedding_2D(embeddings, product_names, method="umap", model_name=model_display_name)

In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.manifold import TSNE
import umap
import random

def visualize_embedding_2D_large(
    embeddings, labels=None, method="umap", model_name="", sample_size=1000, show_text=False
):
    n_samples = embeddings.shape[0]

    # 샘플링
    if n_samples > sample_size:
        indices = random.sample(range(n_samples), sample_size)
        embeddings = embeddings[indices]
        labels = [labels[i] for i in indices] if labels else None

    # 차원 축소
    if method == "umap":
        reducer = umap.UMAP(n_components=2, random_state=42)
    elif method == "tsne":
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    else:
        raise ValueError("Method must be 'umap' or 'tsne'")
    reduced = reducer.fit_transform(embeddings)

    # 시각화
    plt.figure(figsize=(10, 7))
    if labels:
        sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, s=30, palette="tab10", alpha=0.8, linewidth=0)
        plt.legend(title="Label", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], s=30, alpha=0.6, linewidth=0)

    # 선택적으로 텍스트 표시
    if show_text and labels:
        for i in range(len(labels)):
            plt.text(reduced[i, 0] + 0.1, reduced[i, 1], labels[i], fontsize=6, alpha=0.6)

    plt.title(f"{model_name} - {method.upper()} Embedding ({n_samples} → {len(embeddings)})")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [23]:
# 만개 상품명이 있다고 가정할 때
visualize_embedding_2D_large(
    embeddings,
    labels=product_names,
    method="umap",
    model_name="KURE-v1",
    sample_size=1000,   # 1,000개만 시각화
    show_text=False     # 텍스트 생략 → 성능 개선
)


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def get_top_representative_items(embeddings, labels, product_names, top_n=5):
    cluster_to_items = {}

    for cluster_id in set(labels):
        if cluster_id == -1:  # noise 제외
            continue
        cluster_indices = np.where(labels == cluster_id)[0]
        cluster_embeds = embeddings[cluster_indices]
        cluster_names = [product_names[i] for i in cluster_indices]

        centroid = np.mean(cluster_embeds, axis=0, keepdims=True)
        sims = cosine_similarity(centroid, cluster_embeds)[0]
        top_indices = np.argsort(sims)[::-1][:top_n]

        cluster_to_items[cluster_id] = [cluster_names[i] for i in top_indices]

    return cluster_to_items


In [25]:
def visualize_clusters(embeddings, labels, product_names, top_representatives, model_name):
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(embeddings)

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, palette="tab10", s=40, alpha=0.8)

    # 각 클러스터의 대표 1개 상품 (시각화용) 표시
    for cluster_id, item_list in top_representatives.items():
        rep_item = item_list[0]  # 가장 중심 상품 1개
        idx = product_names.index(rep_item)
        x, y = reduced[idx]
        plt.text(x + 0.5, y, f"★ {rep_item}", fontsize=9, color="black", weight="bold")

    plt.title(f"UMAP Clustering with Top Representative Items - {model_name}")
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [32]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
import warnings

warnings.filterwarnings("ignore")

# -----------------------------
# 1. 데이터: 상품명 리스트
# -----------------------------
product_names = [
    "참좋은운전자보험1904", "프리미엄운전자종합보험", "더안심운전자플랜2101", "무배당안전운전자보험2001", "하이카운전자보험플러스",
    "행복한명품암보험", "아이러브건강보험1904", "무배당헬스케어암보험", "암치료집중보장플랜", "더든든건강플랜보험",
    "TOP클래스유니버설종신101종표", "스테이지6대건강종신보험", "평생든든보장종신보험", "라이프플랜종신보험무배당", "무배당통합사망보장보험",
    "튼튼아이보험1906", "프리미엄여성건강보험", "태아부터자녀까지자녀보험", "무배당맘편한어린이보험", "셀럽케어여성암보험",
    "삼성아파트혼합형", "주택화재종합보험", "무배당내집안심화재보험", "부동산임대보장보험", "전세보증금안심보험",
]

# -----------------------------
# 2. 임베딩 함수
# -----------------------------
def get_embedding(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences, convert_to_numpy=True)

# -----------------------------
# 3. 대표 상품 Top-N 추출
# -----------------------------
def get_top_representative_items(embeddings, labels, product_names, top_n=5):
    cluster_to_items = {}
    for cluster_id in set(labels):
        if cluster_id == -1:  # noise 제외
            continue
        cluster_indices = np.where(labels == cluster_id)[0]
        cluster_embeds = embeddings[cluster_indices]
        cluster_names = [product_names[i] for i in cluster_indices]

        centroid = np.mean(cluster_embeds, axis=0, keepdims=True)
        sims = cosine_similarity(centroid, cluster_embeds)[0]
        top_indices = np.argsort(sims)[::-1][:top_n]
        cluster_to_items[cluster_id] = [cluster_names[i] for i in top_indices]
    return cluster_to_items

# -----------------------------
# 4. 시각화 함수 (대표 1개만 표시)
# -----------------------------
def visualize_clusters(embeddings, labels, product_names, top_representatives, model_name):
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(embeddings)

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, palette="tab10", s=40, alpha=0.8)

    # 대표 상품 1개만 강조 텍스트 표시
    for cluster_id, item_list in top_representatives.items():
        rep_item = item_list[0]
        idx = product_names.index(rep_item)
        x, y = reduced[idx]
        plt.text(x + 0.5, y, f"★ {rep_item}", fontsize=9, color="black", weight="bold")

    plt.title(f"UMAP Clustering with Top Representative Items - {model_name}")
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# -----------------------------
# 5. 전체 실행
# -----------------------------
model_name = "intfloat/multilingual-e5-large"  # SentenceTransformer 모델 ID
embeddings = get_embedding(model_name, product_names)

# 클러스터링
clusterer = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=2)
labels = clusterer.fit_predict(embeddings)

# 대표 상품 Top-N 추출
top_representatives = get_top_representative_items(embeddings, labels, product_names, top_n=5)

# 대표 상품 리스트 출력
print("📌 클러스터별 대표 상품 (Top 5):")
for cluster_id, items in sorted(top_representatives.items()):
    print(f"\nCluster {cluster_id} 대표 상품 Top 5:")
    for rank, item in enumerate(items, 1):
        print(f"  {rank}. {item}")

# 시각화
visualize_clusters(embeddings, labels, product_names, top_representatives, model_name)